# 04. 5주차 데이터 추출 (Week 5)

기존 28일(2/15~3/14) 데이터에 1주일(3/15~3/21)을 추가하여 **총 35일(5주)** 데이터를 확보합니다.

- Dry run으로 비용 확인 → 실제 추출 → 검증

## 1. 설정

In [1]:
%%time
import os
from pathlib import Path
from datetime import date

from gharchive.client import create_client, get_logger
from gharchive.extract import dry_run, extract_date_range

NOTEBOOK_DIR = Path(__file__).parent if "__file__" in dir() else Path.cwd()
PROJECT_ROOT = Path("/Users/kakao/bda-2")
KEY_PATH = os.environ.get("GCP_KEY_PATH", str(PROJECT_ROOT / "gcp-key.json"))
OUTPUT_DIR = PROJECT_ROOT / "data" / "daily_agg"

client = create_client(KEY_PATH)
logger = get_logger("extract")

START_DATE = date(2026, 3, 15)
END_DATE = date(2026, 3, 21)

print("Client ready:", client.project)
print(f"추출 범위: {START_DATE} ~ {END_DATE} ({(END_DATE - START_DATE).days + 1}일)")

Client ready: bda-coai
추출 범위: 2026-03-15 ~ 2026-03-21 (7일)
CPU times: user 1.02 s, sys: 245 ms, total: 1.26 s
Wall time: 1.45 s


## 2. Dry Run (비용 확인)

In [2]:
%%time
from datetime import timedelta

current = START_DATE
total_bytes = 0
total_cost = 0

print(f"{'날짜':<12} {'스캔량':>8} {'비용(USD)':>10}")
print("-" * 32)

while current <= END_DATE:
    date_str = current.strftime("%Y%m%d")
    info = dry_run(client, date_str)
    gb = info["bytes_processed"] / 1024**3
    cost = info["estimated_cost_usd"]
    total_bytes += info["bytes_processed"]
    total_cost += cost
    print(f"{date_str:<12} {gb:>7.2f} GB  ${cost:>.4f}")
    current += timedelta(days=1)

print("-" * 32)
print(f"{'합계':<12} {total_bytes / 1024**3:>7.2f} GB  ${total_cost:>.4f}")

날짜                스캔량    비용(USD)
--------------------------------


20260315        0.10 GB  $0.0006


20260316        0.10 GB  $0.0006


20260317        0.10 GB  $0.0006


20260318        0.10 GB  $0.0006


20260319        0.10 GB  $0.0006


20260320        0.10 GB  $0.0006


20260321        0.10 GB  $0.0006
--------------------------------
합계              0.68 GB  $0.0042
CPU times: user 51.3 ms, sys: 9.09 ms, total: 60.4 ms
Wall time: 4.42 s


## 3. 추출 실행

In [3]:
%%time
saved_files = extract_date_range(client, START_DATE, END_DATE, OUTPUT_DIR, logger)
print(f"\n저장된 파일: {len(saved_files)}개")

22:04:59 INFO Extracting 20260315 ...


22:05:04 INFO   → 1,261,017 rows, 14208 KB


22:05:04 INFO Extracting 20260316 ...


22:05:09 INFO   → 1,455,041 rows, 16232 KB


22:05:09 INFO Extracting 20260317 ...


...

22:05:23 INFO Extracting 20260320 ...


22:05:27 INFO   → 1,356,418 rows, 15299 KB


22:05:28 INFO Extracting 20260321 ...


22:05:33 INFO   → 1,253,736 rows, 14092 KB



저장된 파일: 7개
CPU times: user 2.72 s, sys: 764 ms, total: 3.48 s
Wall time: 34 s


## 4. 검증

전체 35일 데이터가 정상적으로 존재하는지 확인합니다.

In [4]:
%%time
import pandas as pd
from gharchive.transform import optimize_types

parquet_files = sorted(OUTPUT_DIR.glob("*.parquet"))
print(f"Parquet 파일 수: {len(parquet_files)}")
print(f"범위: {parquet_files[0].stem} ~ {parquet_files[-1].stem}")

# 신규 7일분 요약
new_files = [f for f in parquet_files if f.stem >= "20260315"]
for f in new_files:
    df = pd.read_parquet(f)
    print(f"  {f.stem}: {len(df):>10,} rows, {f.stat().st_size / 1024:,.0f} KB")

Parquet 파일 수: 35
범위: 20260215 ~ 20260321
  20260315:  1,261,017 rows, 14,208 KB
  20260316:  1,455,041 rows, 16,232 KB
  20260317:  1,450,081 rows, 16,374 KB
  20260318:  1,422,586 rows, 16,052 KB
  20260319:  1,402,810 rows, 15,864 KB


  20260320:  1,356,418 rows, 15,299 KB
  20260321:  1,253,736 rows, 14,092 KB
CPU times: user 432 ms, sys: 67.2 ms, total: 499 ms
Wall time: 247 ms
